In [ ]:
%pip install -q requests sounddevice numpy pandas
import getpass, os, wave
from pathlib import Path
import numpy as np
import sounddevice as sd
from IPython.display import Audio, display

REFERENCE = 'Исключить задачи с приоритетом низкий.\
      Сгруппировать по исполнителям и найти открытые задачи без дедлайна.'
SAMPLE_RATE = 16_000
RECORD_SECONDS = 10
AUDIO_PATH = Path('asr_benchmark.wav')
ROUTERAI_API_KEY = os.getenv('ROUTERAI_API_KEY') or getpass.getpass('ROUTERAI_API_KEY: ')

print('Произнесите ТОЧНО эту фразу:')
print(REFERENCE)
input(f'Нажмите Enter для начала записи на {RECORD_SECONDS} секунд...')
audio = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
sd.wait()
pcm = (np.clip(audio[:, 0], -1, 1) * 32767).astype('<i2')
with wave.open(str(AUDIO_PATH), 'wb') as wav:
    wav.setnchannels(1); wav.setsampwidth(2); wav.setframerate(SAMPLE_RATE); wav.writeframes(pcm.tobytes())
print(f'Сохранено: {AUDIO_PATH.resolve()} ({AUDIO_PATH.stat().st_size} bytes)')
display(Audio(str(AUDIO_PATH)))


In [ ]:
import re, time
import pandas as pd
import requests

MODELS = [
    'nvidia/nemotron-3.5-asr-streaming-multilingual-0.6b',
    'qwen/qwen3-asr-0.6b',
]
URL = 'https://routerai.ru/api/v1/audio/transcriptions'

def words(text):
    return re.findall(r'[0-9a-zа-яё]+', text.lower().replace('ё', 'е'))

def wer(reference, hypothesis):
    ref, hyp = words(reference), words(hypothesis)
    dp = list(range(len(hyp) + 1))
    for i, rw in enumerate(ref, 1):
        prev, dp[0] = dp[0], i
        for j, hw in enumerate(hyp, 1):
            old = dp[j]
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev + (rw != hw))
            prev = old
    return dp[-1] / max(1, len(ref))

rows = []
for model in MODELS:
    started = time.perf_counter()
    with AUDIO_PATH.open('rb') as audio_file:
        response = requests.post(URL, headers={'Authorization': f'Bearer {ROUTERAI_API_KEY}'},
                                 data={'model': model, 'language': 'ru'},
                                 files={'file': (AUDIO_PATH.name, audio_file, 'audio/wav')}, timeout=60)
    latency = time.perf_counter() - started
    response.raise_for_status()
    data = response.json()
    transcript = (data.get('text') or data.get('data', {}).get('text') or '').strip()
    rows.append({'model': model, 'latency_s': round(latency, 3), 'WER': round(wer(REFERENCE, transcript), 4),
                 'exact_normalized': words(REFERENCE) == words(transcript), 'transcript': transcript})

pd.DataFrame(rows).sort_values(['WER', 'latency_s']).reset_index(drop=True)
